# Exploratory Data Analysis: ABS Building Approvals

This notebook explores the ABS 8731.0 quarterly dwelling approvals data.
Key questions:
- What is the seasonal pattern across LGAs?
- How does state-level variation look?
- Where is the 2022 structural break visible in the raw series?

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

FEATURES_PATH = Path('../data/processed/features.parquet')

## 1. Load feature dataset

In [ ]:
df = pd.read_parquet(FEATURES_PATH)
df['quarter'] = pd.PeriodIndex(df['quarter'], freq='Q')
print(f"Shape: {df.shape}")
print(f"Date range: {df['quarter'].min()} to {df['quarter'].max()}")
print(f"LGAs: {df['lga_code'].nunique()}")
df.head()

## 2. National aggregate: quarterly approvals over time

In [ ]:
national = df.groupby('quarter')['dwellings_approved'].sum().reset_index()
national['quarter_dt'] = national['quarter'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(national['quarter_dt'], national['dwellings_approved'], color='steelblue')
ax.axvline(pd.Timestamp('2022-07-01'), color='red', linestyle='--', label='Housing Accord (Aug 2022)')
ax.set_title('National Quarterly Dwelling Approvals')
ax.set_ylabel('Total dwellings approved')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Year-on-year approvals trend

The ABS Regional dataset is **annual** (one observation per LGA per financial year, mapped to Q2). Within-year seasonal decomposition is not applicable. The chart below shows how average approvals per LGA have shifted year-by-year, with the post-Accord period highlighted.

In [ ]:
by_year = (
    df.groupby(df['quarter'].apply(lambda q: q.start_time.year))['dwellings_approved']
    .mean()
    .reset_index()
)
by_year.columns = ['year', 'mean_approvals']

colors = ['#F44336' if yr >= 2023 else 'steelblue' for yr in by_year['year']]
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(by_year['year'].astype(str), by_year['mean_approvals'],
              color=colors, edgecolor='white')
ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
ax.axvline(2.5, color='red', linestyle='--', alpha=0.6, label='Housing Accord era begins')
ax.set_title('Mean Approvals per LGA by Financial Year  (red = post-Accord)')
ax.set_ylabel('Mean dwellings approved')
ax.set_xlabel('Financial year ending (calendar year)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Construction cost pressure vs approvals: the 2022 policy-shift break

In [ ]:
macro = df.drop_duplicates('quarter').sort_values('quarter')[
    ['quarter', 'construction_cost_yoy', 'post_accord_2022']
].copy()
macro['quarter_dt'] = macro['quarter'].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.plot(national['quarter_dt'], national['dwellings_approved'], color='steelblue', label='Approvals (left)')
ax2.plot(macro['quarter_dt'], macro['construction_cost_yoy'] * 100, color='darkorange',
         linestyle='--', label='Construction cost YoY % (right)')

ax1.axvline(pd.Timestamp('2022-07-01'), color='red', linestyle=':', linewidth=2,
            label='Housing Accord (Aug 2022)')
ax1.set_title('Dwelling Approvals vs House Construction Cost Pressure')
ax1.set_ylabel('Dwellings approved', color='steelblue')
ax2.set_ylabel('House construction cost YoY (%)', color='darkorange')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
plt.tight_layout()
plt.show()

## 5. Class distribution: approvals across LGAs

In [ ]:
from data.pipeline import describe
describe(df)